<a href="https://colab.research.google.com/github/christianlocher/llmrag/blob/main/I_Text_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Packages

In [5]:
!pip install -U pypdf langchain-community

import requests
from io import BytesIO
from langchain_community.document_loaders import PyPDFLoader

#I Text Chunking

We will try some of the most used text splitters here with a pdf loaded from the web.

Please use for first tests: Use the provided URL for a PDF and load its content:

In [35]:
def load_pdf_from_url(url: str):
    response = requests.get(url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    # Save the PDF content to a temporary file
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
        tmp_file.write(response.content)
        tmp_file_path = tmp_file.name

    loader = PyPDFLoader(tmp_file_path)
    documents = loader.load()

    # Clean up the temporary file after loading
    os.remove(tmp_file_path)

    return documents

import requests
from io import BytesIO
import tempfile
from langchain_community.document_loaders import PyPDFLoader
import os

# pdf_url = "https://www.arbonia.com/fileadmin/api/tpo/wp-content/uploads/2019/08/13_Basisnormen-f%C3%BCr-Tischler-und-Schreiner.pdf"
pdf_url = "https://www.scanlife.de/wp-content/uploads/2025/10/web_ES_ScanLife_Mag_25_100S_03_RZ.pdf"
documents_from_url = load_pdf_from_url(pdf_url)
print(f"Loaded {len(documents_from_url)} pages from URL.")
if documents_from_url:
    print(documents_from_url[0].page_content[:500]) # Print first 500 characters of the first page

Loaded 100 pages from URL.
Lifestyle 25 I 26


Please use for further tests with more and different documents

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

DATA_PATH = r"sample_data"
#DATA_PATH = '/content/drive/MyDrive/YourFolderName/' # Use to place files on G-Drive

loader = PyPDFDirectoryLoader(DATA_PATH)

### Other options:
# PDF Loader
# https://medium.com/data-and-beyond/document-loaders-in-langchain-f23d3ce70d66
# Simple, clean PDFs: Use PyPDFLoader
# PDFs with tables/columns: Use PDFPlumberLoader
# Scanned/image PDFs: Use UnstructuredPDFLoader or AmazonTextractPDFLoader
# Need layout and image data: Use PyMuPDFLoader
# Want best structure extraction: Use UnstructuredPDFLoader

documents_from_url = loader.load()


#I-1 CharacterTextSplitter

In [27]:
from langchain_text_splitters import CharacterTextSplitter

regex_separator = r"(?=\n(?:Seite|Wolfgang Heer|§)\s*\d+)"
text_splitter = CharacterTextSplitter(
    #separator="\n\n", # Page break
    #separator=regex_separator, # Separates on Regex - Turn "is_separator_regex" = True and activate keep_separator=True
    separator=".", # Sentence end
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=True,
    keep_separator=True
)

# Info on separators: https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter

chunks = text_splitter.split_documents(documents_from_url)

i = 0

for chunk in chunks:
    print(chunk.page_content)
    print('####################################################################')

    i += 1

Basisnormen für Tischler und Schreiner 
Fachregeln 
Normen gelten nicht selten als „allgemein anerkannte 
Regel der Technik (aaRdT)". Daher ist es notwendig, 
jene Normen zu kennen, die für die jeweiligen Auf-
träge relevant sind. 
Mit dieser kleinen Zusammenstellung soll ein grober 
Überblick zu wichtigen Normen für Tischler und 
Schreiner gegeben werden. Da Normen dem Copy-
right unterliegen, ist ein Abdruck von Normen an die-
ser Stelle nicht möglich. Normen können über 
www.beuth.de  bezogen
####################################################################
möglich. Normen können über 
www.beuth.de  bezogen werden. Sinnvoll ist dabei 
meist die Anschaffung von DIN-Taschenbüchern, in 
welchen Normen nach Themen zusammengestellt 
sind. So gibt es z. B. das „Normenhandbuch Tischler-
und Schreinerhandwerk" in welchem viele Normen 
abgebildet sind. In den DIN-Taschenbüchern 359 und 
360 finden sich z. B. Nomen für die Holzwirtschaft. 
Wer viel mit Normen arbeiten muss, kann sich auch 

#I-2 RecursiveCharacterTextSplitter

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    #separators=["\n\n"], #split at paragraph = 2 new lines
    #separators=[r"(?<=[.!?]) +"], #split at any end of a sentence
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False, # use the separators above or not
)

chunks = text_splitter.split_documents(documents_from_url)

i = 0

for chunk in chunks:
    print(chunk.page_content)
    print('####################################################################')

    i += 1

Basisnormen für Tischler und Schreiner 
Fachregeln 
Normen gelten nicht selten als „allgemein anerkannte 
Regel der Technik (aaRdT)". Daher ist es notwendig, 
jene Normen zu kennen, die für die jeweiligen Auf-
träge relevant sind. 
Mit dieser kleinen Zusammenstellung soll ein grober 
Überblick zu wichtigen Normen für Tischler und 
Schreiner gegeben werden. Da Normen dem Copy-
right unterliegen, ist ein Abdruck von Normen an die-
ser Stelle nicht möglich. Normen können über
####################################################################
ser Stelle nicht möglich. Normen können über 
www.beuth.de  bezogen werden. Sinnvoll ist dabei 
meist die Anschaffung von DIN-Taschenbüchern, in 
welchen Normen nach Themen zusammengestellt 
sind. So gibt es z. B. das „Normenhandbuch Tischler-
und Schreinerhandwerk" in welchem viele Normen 
abgebildet sind. In den DIN-Taschenbüchern 359 und 
360 finden sich z. B. Nomen für die Holzwirtschaft. 
Wer viel mit Normen arbeiten muss, kann sich auch 
besti

#I-3 SpacyTextSplitter

This section here is different from others, as it adds some more metadata to the documents to show how this can be used later. The section that adds metadata can be used independent from SpacyTextSplitter with other textsplitters.

In [36]:
from langchain_text_splitters import SpacyTextSplitter
import spacy.cli
from langchain_core.documents import Document
from datetime import datetime

spacy.cli.download("de_core_news_sm")
# be careful to use the right dictionary, see: https://spacy.io/models
text_splitter = SpacyTextSplitter(
    pipeline="de_core_news_sm",
    chunk_size=500,
    chunk_overlap=50
    )

rawtext = text_splitter.split_documents(documents_from_url)

chunks = []

for i, doc_chunk in enumerate(rawtext):
    # Create a new Document object with added metadata
    new_metadata = {
        "chunk_id": i,
        "source": "",
        "orig_source": "Anysource",
        "mandant": "OtherCompany 123",
        "last_updated": datetime.now().isoformat(),
        "language": "de",
        **doc_chunk.metadata # Include existing metadata from the chunk
    }
    doc = Document(
        page_content=doc_chunk.page_content, # Correctly access content from the current Document object
        metadata=new_metadata
    )
    chunks.append(doc)


i = 0

for chunk in chunks:
    print(chunk)
    print('####################################################################')

    i += 1

✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


page_content='Lifestyle 25 I 26' metadata={'chunk_id': 0, 'source': '/tmp/tmpr4tibe40.pdf', 'orig_source': 'Anysource', 'mandant': 'OtherCompany 123', 'last_updated': '2026-05-14T10:15:04.386011', 'language': 'de', 'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.5 (Macintosh)', 'creationdate': '2025-09-16T14:31:45+02:00', 'moddate': '2025-09-16T14:38:12+02:00', 'trapped': '/False', 'total_pages': 100, 'page': 0, 'page_label': '1'}
####################################################################
page_content='389.-
1. Couch- 
tisch


Mit organischer  
Baumkante
MODERN
Mitten im Leben!
Wunderschöne Couch-  
und Beistelltische in der angesagten 
Kombination massive Wildeiche  
mit schwarzem Metall.


1. Couchtisch Platte: Wildeiche massiv,  
natur geölt, 25 mm stark mit Baumkante,  
Gestell: schräge Metallfüße, 4 mm stark, schwarz  
pulverbeschichtet, B/H/T: 110 x 40 x 60 cm: | 389.- | 
2. Beistelltische, 2-er Set' metadata={'chunk_id': 1, 'source': '/tmp/tmpr4tibe

#I-4 RecursiveCharacterSplitter with Tiktoken (Token-based instead of Char-based Chunking)


In [33]:
!pip install -q tiktoken langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base") # Use this for GPT-4o / GPT-4, see: https://developers.openai.com/cookbook/examples/how_to_count_tokens_with_tiktoken/

def tic_token_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)


text_splitter = RecursiveCharacterTextSplitter(
    #separators=["\n\n"], #split at paragraph = 2 new lines
    separators=[r"(?<=[.!?]) +"], #split at any end of a sentence
    chunk_size=500,
    chunk_overlap=50,
    length_function=tic_token_len,
    is_separator_regex=False, # use the separators above or not
)

chunks = text_splitter.split_documents(documents_from_url)

i = 0

for chunk in chunks:
    print(chunk.page_content)
    print('####################################################################')

    i += 1

Basisnormen für Tischler und Schreiner 
Fachregeln 
Normen gelten nicht selten als „allgemein anerkannte 
Regel der Technik (aaRdT)". Daher ist es notwendig, 
jene Normen zu kennen, die für die jeweiligen Auf-
träge relevant sind. 
Mit dieser kleinen Zusammenstellung soll ein grober 
Überblick zu wichtigen Normen für Tischler und 
Schreiner gegeben werden. Da Normen dem Copy-
right unterliegen, ist ein Abdruck von Normen an die-
ser Stelle nicht möglich. Normen können über 
www.beuth.de  bezogen werden. Sinnvoll ist dabei 
meist die Anschaffung von DIN-Taschenbüchern, in 
welchen Normen nach Themen zusammengestellt 
sind. So gibt es z. B. das „Normenhandbuch Tischler-
und Schreinerhandwerk" in welchem viele Normen 
abgebildet sind. In den DIN-Taschenbüchern 359 und 
360 finden sich z. B. Nomen für die Holzwirtschaft. 
Wer viel mit Normen arbeiten muss, kann sich auch 
bestimmter DIN-Portale bedienen, auf welchen man 
zahlreiche Normen herunterladen kann. Für die Nut-
zung der Portale f

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

After mounting your Google Drive, you can access your files. Please specify the path to your data in Google Drive. For example:

```python
data_path_in_drive = '/content/drive/MyDrive/YourFolderName/your_data.csv'
# Now you can use this path to load your data, e.g., with pandas:
# import pandas as pd
# df = pd.read_csv(data_path_in_drive)
# display(df.head())
```